# NutriMatch: Advanced Food Recommendation Model (7-Day Meal Plan)
Notebook ini mengimplementasikan model Deep Learning NutriMatch yang diperbarui dengan:
1. **Dataset v2** (`train_ready_dataset_v2.csv`) — skema kolom yang diperluas dengan filter menu-ready, konteks halal, `recommendation_item_type`, dan meal-time suitability.
2. **Meal-Time Filtering** — filter makanan berdasarkan kolom `suitable_breakfast`, `suitable_lunch`, `suitable_dinner`.
3. **Variety Penalty** — mencegah makanan yang sama muncul berulang dalam 7 hari.
4. **7-Day Meal Plan** — menghasilkan jadwal makan lengkap 7 hari (Pagi, Siang, Malam).
5. **Rule-Based Guardrails** — keamanan alergi absolut, filter makanan siap rekomendasi, dan filter halal opsional sebelum rekomendasi final.

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from datetime import datetime, timedelta
import os

# Set seed for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 1. Load Data & Parse Features (Dataset v2)

In [2]:
# ============================================================
# LOAD DATASET TERBARU (train_ready_dataset_v2.csv)
# Kolom baru yang dimanfaatkan:
#   - suitable_breakfast, suitable_lunch, suitable_dinner  → meal-time filter
#   - primary_meal_time                                   → fallback meal label
#   - cooking_category, main_ingredient                   → info tambahan
#   - is_recommendable_food, recommendation_item_type      → filter menu-ready
#   - halal_status, is_halal_candidate                    → guardrail halal
# ============================================================
DATASET_PATH = 'Data/train_ready_dataset_v2.csv'
ENFORCE_HALAL_FILTER = True

food_df = pd.read_csv(DATASET_PATH)
user_df = pd.read_csv('Data/user_profile_features_schema.csv')

print(f"Dataset path     : {DATASET_PATH}")
print(f"Food data shape  : {food_df.shape}")
print(f"User data shape  : {user_df.shape}")
print(f"\nKolom Food Dataset:")
print(food_df.columns.tolist())

food_subset = food_df.copy()
user_subset = user_df.copy()

# ----------------------------------------------------------
# Filter khusus dataset v2: hanya makanan yang siap direkomendasikan
# dan aman untuk konteks halal jika ENFORCE_HALAL_FILTER=True.
# ----------------------------------------------------------
def normalize_bool_col(series, default=False):
    return series.astype(str).str.lower().map(
        {'true': True, 'false': False, '1': True, '0': False}
    ).fillna(default).astype(bool)

before_v2_filter = len(food_subset)

if 'is_recommendable_food' in food_subset.columns:
    food_subset['is_recommendable_food'] = normalize_bool_col(food_subset['is_recommendable_food'], default=True)
    food_subset = food_subset[food_subset['is_recommendable_food']]

for col in ['ingredient_only_flag', 'raw_ingredient_flag', 'contains_non_halal_ingredient', 'is_halal_candidate']:
    if col in food_subset.columns:
        default_value = True if col == 'is_halal_candidate' else False
        food_subset[col] = normalize_bool_col(food_subset[col], default=default_value)

if 'ingredient_only_flag' in food_subset.columns:
    food_subset = food_subset[~food_subset['ingredient_only_flag']]
if 'raw_ingredient_flag' in food_subset.columns:
    food_subset = food_subset[~food_subset['raw_ingredient_flag']]

if ENFORCE_HALAL_FILTER:
    if 'contains_non_halal_ingredient' in food_subset.columns:
        food_subset = food_subset[~food_subset['contains_non_halal_ingredient']]
    if 'is_halal_candidate' in food_subset.columns:
        food_subset = food_subset[food_subset['is_halal_candidate']]
    if 'halal_status' in food_subset.columns:
        food_subset = food_subset[food_subset['halal_status'].fillna('halal_candidate').eq('halal_candidate')]

food_subset = food_subset.reset_index(drop=True)
print(f"Food after v2 guardrails: {food_subset.shape} (removed {before_v2_filter - len(food_subset)} rows)")

# ----------------------------------------------------------
# Parse user allergy vector: "0,0,0,0,0,0,0,0" → 8 kolom
# ----------------------------------------------------------
allergy_cols_user = [f'user_allergy_{i}' for i in range(8)]
user_allergies = user_subset['allergy_vector'].str.split(',', expand=True).astype(np.float32)
user_allergies.columns = allergy_cols_user

# SYNTHETIC ALLERGY INJECTION (untuk training)
np.random.seed(42)
random_allergies = np.random.choice([0.0, 1.0], size=user_allergies.shape, p=[0.8, 0.2])
user_allergies = pd.DataFrame(
    np.maximum(user_allergies.values, random_allergies),
    columns=allergy_cols_user
)
user_subset = pd.concat([user_subset, user_allergies], axis=1)

# ----------------------------------------------------------
# Peta alergi makanan — 8 kolom standar
# ----------------------------------------------------------
allergy_cols_food = [
    'contains_gluten', 'contains_dairy', 'contains_nuts', 'contains_peanut',
    'contains_seafood', 'contains_egg', 'contains_soy', 'contains_celery'
]
for col in allergy_cols_food:
    if col in food_subset.columns:
        food_subset[col] = food_subset[col].astype(str).str.lower().map(
            {'true': 1.0, 'false': 0.0, '1': 1.0, '0': 0.0}
        ).fillna(0.0).astype(np.float32)
    else:
        food_subset[col] = 0.0

# ----------------------------------------------------------
# Normalisasi kolom suitable_* (bool / string → int)
# ----------------------------------------------------------
for col in ['suitable_breakfast', 'suitable_lunch', 'suitable_dinner']:
    if col in food_subset.columns:
        food_subset[col] = food_subset[col].astype(str).str.lower().map(
            {'true': 1, 'false': 0, '1': 1, '0': 0}
        ).fillna(0).astype(int)
    else:
        food_subset[col] = 1  # default: cocok semua waktu makan

print("\nContoh data makanan (5 baris pertama):")
preview_cols = [
    'food_name', 'calories_100g', 'food_category', 'recommendation_item_type',
    'suitable_breakfast', 'suitable_lunch', 'suitable_dinner',
    'contains_gluten', 'contains_seafood', 'halal_status'
]
preview_cols = [col for col in preview_cols if col in food_subset.columns]
print(food_subset[preview_cols].head())

Dataset path     : Data/train_ready_dataset_v2.csv
Food data shape  : (975, 67)
User data shape  : (100, 16)

Kolom Food Dataset:
['food_id', 'food_name', 'food_name_clean', 'calories_100g', 'protein_100g', 'fat_100g', 'carbohydrate_100g', 'image_url', 'source', 'zero_calorie_flag', 'calorie_inconsistent_flag', 'calorie_macro_diff', 'cooking_category', 'main_ingredient', 'meal_time', 'contains_gluten', 'contains_dairy', 'contains_nuts', 'contains_peanut', 'contains_seafood', 'contains_egg', 'contains_soy', 'contains_celery', 'contains_mustard', 'contains_sesame', 'contains_sulfite', 'contains_other', 'contains_unknown', 'confidence', 'allergen_sources', 'label_sources', 'allergen_match_count', 'merge_status', 'food_category', 'food_category_source', 'base_ingredient', 'base_ingredient_tags', 'recipe_reference_match', 'recipe_reference_source', 'recipe_reference_title', 'recipe_ingredients_reference', 'suitable_breakfast', 'suitable_lunch', 'suitable_dinner', 'meal_time_tags', 'primary_

## 2. Data Preprocessing & Synthetic Pairs (Train/Val Split)

In [3]:
user_macro_cols = ['target_calorie', 'protein_target_g', 'fat_target_g', 'carb_target_g']
food_macro_cols = ['calories_100g', 'protein_100g', 'fat_100g', 'carbohydrate_100g']

user_subset[user_macro_cols] = user_subset[user_macro_cols].fillna(0)
food_subset[food_macro_cols] = food_subset[food_macro_cols].fillna(0)

# Cartesian product
user_subset['dummy_key'] = 1
food_subset['dummy_key'] = 1
pairs = pd.merge(user_subset, food_subset, on='dummy_key').drop('dummy_key', axis=1)

# Label 1: Match Score (Regression)
calorie_diff = np.abs((pairs['target_calorie'] / 3) - pairs['calories_100g'])
pairs['match_score'] = np.exp(-calorie_diff / 500).astype(np.float32)

# Label 2: Allergy Risk (Classification)
user_all_arr = pairs[allergy_cols_user].values
food_all_arr = pairs[allergy_cols_food].values
pairs['is_allergic'] = np.any(
    np.logical_and(user_all_arr == 1, food_all_arr == 1), axis=1
).astype(np.float32)

# Inputs
X_user_macro   = pairs[user_macro_cols].values.astype(np.float32)
X_food_macro   = pairs[food_macro_cols].values.astype(np.float32)
X_user_allergy = pairs[allergy_cols_user].values.astype(np.float32)
X_food_allergy = pairs[allergy_cols_food].values.astype(np.float32)

y_match   = pairs['match_score'].values
y_allergy = pairs['is_allergic'].values

# Normalisasi makro
user_max = np.max(X_user_macro, axis=0) + 1e-9
food_max = np.max(X_food_macro, axis=0) + 1e-9
X_user_macro = X_user_macro / user_max
X_food_macro = X_food_macro / food_max

# Shuffle & Split 80/20
total_samples = len(pairs)
indices = np.random.permutation(total_samples)
train_idx = indices[:int(0.8 * total_samples)]
val_idx   = indices[int(0.8 * total_samples):]

def create_dataset(idx):
    return tf.data.Dataset.from_tensor_slices((
        (X_user_macro[idx], X_user_allergy[idx],
         X_food_macro[idx], X_food_allergy[idx]),
        (y_match[idx], y_allergy[idx])
    )).batch(64)

train_dataset = create_dataset(train_idx)
val_dataset   = create_dataset(val_idx)

print(f"Total pairs : {total_samples} (Train: {len(train_idx)}, Val: {len(val_idx)})")
print(f"Allergic pairs (train): {np.sum(y_allergy[train_idx])}")

Total pairs : 95800 (Train: 76640, Val: 19160)
Allergic pairs (train): 2513.0


## 3. Multi-Head Architecture & Custom Layer

In [4]:
# Custom Interaction Layer
class InteractionLayer(tf.keras.layers.Layer):
    def call(self, inputs):
        a, b = inputs
        return a * b  # Element-wise interaction

# Custom Asymmetric Allergy Loss
@tf.function
def asymmetric_allergy_loss(y_true, y_pred):
    """Penalizes false negatives more heavily (10x) than false positives."""
    y_true = tf.cast(tf.reshape(y_true, [-1, 1]), tf.float32)
    fn_weight = 10.0  # sangat mahal jika melewatkan alergi
    fp_weight = 1.0
    weights = y_true * fn_weight + (1 - y_true) * fp_weight
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return tf.reduce_mean(weights[:, 0] * bce)

def build_nutrimatch_model():
    # Input layers
    inp_user_macro   = tf.keras.Input(shape=(4,), name='user_macro')
    inp_food_macro   = tf.keras.Input(shape=(4,), name='food_macro')
    inp_user_allergy = tf.keras.Input(shape=(8,), name='user_allergy')
    inp_food_allergy = tf.keras.Input(shape=(8,), name='food_allergy')

    # Macro towers
    u_emb = tf.keras.layers.Dense(16, activation='relu')(inp_user_macro)
    f_emb = tf.keras.layers.Dense(16, activation='relu')(inp_food_macro)

    # Macro interaction head → Match Score
    macro_interaction = InteractionLayer()([u_emb, f_emb])
    match_out = tf.keras.layers.Dense(1, activation='sigmoid', name='match_output')(macro_interaction)

    # Allergy head → Allergy Risk
    allergy_concat = tf.keras.layers.Concatenate()([inp_user_allergy, inp_food_allergy])
    allergy_hidden = tf.keras.layers.Dense(16, activation='relu')(allergy_concat)
    allergy_out = tf.keras.layers.Dense(1, activation='sigmoid', name='allergy_output')(allergy_hidden)

    model = tf.keras.Model(
        inputs=[inp_user_macro, inp_user_allergy, inp_food_macro, inp_food_allergy],
        outputs=[match_out, allergy_out]
    )
    return model

model = build_nutrimatch_model()
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_macro          │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ food_macro          │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_allergy        │ (None, 8)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ food_allergy        │ (None, 8)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 16)        │         80 │ user_macro[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 16)        │         80 │ food_macro[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 16)        │          0 │ user_allergy[0][… │
│ (Concatenate)       │                   │            │ food_allergy[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ interaction_layer   │ (None, 16)        │          0 │ dense[0][0],      │
│ (InteractionLayer)  │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 16)        │        272 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ match_output        │ (None, 1)         │         17 │ interaction_laye… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ allergy_output      │ (None, 1)         │         17 │ dense_2[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 466 (1.82 KB)

 Trainable params: 466 (1.82 KB)

 Non-trainable params: 0 (0.00 B)

## 4. Custom Training Loop dengan Metrics

In [5]:
optimizer   = tf.keras.optimizers.Adam(learning_rate=1e-3)
mse_loss_fn = tf.keras.losses.MeanSquaredError()

log_dir = 'logs/nutrimatch_' + datetime.now().strftime('%Y%m%d-%H%M%S')
summary_writer = tf.summary.create_file_writer(log_dir)

train_mae    = tf.keras.metrics.MeanAbsoluteError()
train_recall = tf.keras.metrics.Recall(thresholds=0.5)
val_mae      = tf.keras.metrics.MeanAbsoluteError()
val_recall   = tf.keras.metrics.Recall(thresholds=0.5)

epochs = 5
for epoch in range(epochs):
    print(f"\n--- Epoch {epoch+1}/{epochs} ---")

    for x_batch, y_batch in train_dataset:
        with tf.GradientTape() as tape:
            match_pred, allergy_pred = model(x_batch, training=True)
            loss_match   = mse_loss_fn(tf.expand_dims(y_batch[0], -1), match_pred)
            loss_allergy = asymmetric_allergy_loss(y_batch[1], allergy_pred)
            total_loss   = loss_match + loss_allergy
        grads = tape.gradient(total_loss, model.trainable_weights)
        optimizer.apply_gradients(zip(grads, model.trainable_weights))
        train_mae.update_state(tf.expand_dims(y_batch[0], -1), match_pred)
        train_recall.update_state(tf.expand_dims(y_batch[1], -1), allergy_pred)

    for x_val, y_val in val_dataset:
        val_match_pred, val_allergy_pred = model(x_val, training=False)
        val_mae.update_state(tf.expand_dims(y_val[0], -1), val_match_pred)
        val_recall.update_state(tf.expand_dims(y_val[1], -1), val_allergy_pred)

    t_mae, t_rec = train_mae.result(), train_recall.result()
    v_mae, v_rec = val_mae.result(), val_recall.result()
    print(f"Train -> MAE: {t_mae:.4f} | Allergy Recall: {t_rec:.4f}")
    print(f"Val   -> MAE: {v_mae:.4f} | Allergy Recall: {v_rec:.4f}")

    with summary_writer.as_default():
        tf.summary.scalar('Train_MAE',    t_mae, step=epoch)
        tf.summary.scalar('Train_Recall', t_rec, step=epoch)
        tf.summary.scalar('Val_MAE',      v_mae, step=epoch)
        tf.summary.scalar('Val_Recall',   v_rec, step=epoch)

    train_mae.reset_state(); train_recall.reset_state()
    val_mae.reset_state();   val_recall.reset_state()


--- Epoch 1/5 ---
Train -> MAE: 0.0664 | Allergy Recall: 0.5444
Val   -> MAE: 0.0208 | Allergy Recall: 0.9823

--- Epoch 2/5 ---
Train -> MAE: 0.0197 | Allergy Recall: 0.9805
Val   -> MAE: 0.0179 | Allergy Recall: 0.9936

--- Epoch 3/5 ---
Train -> MAE: 0.0181 | Allergy Recall: 0.9920
Val   -> MAE: 0.0174 | Allergy Recall: 1.0000

--- Epoch 4/5 ---
Train -> MAE: 0.0178 | Allergy Recall: 0.9964
Val   -> MAE: 0.0169 | Allergy Recall: 1.0000

--- Epoch 5/5 ---
Train -> MAE: 0.0173 | Allergy Recall: 0.9972
Val   -> MAE: 0.0165 | Allergy Recall: 1.0000


## 5. Save Model

In [6]:
model.save('nutrimatch_model.keras')
print("Model tersimpan sebagai 'nutrimatch_model.keras'")

Model tersimpan sebagai 'nutrimatch_model.keras'


## 6. Inference & 7-Day Meal Plan

**Fitur baru:**
- Filter `suitable_breakfast / suitable_lunch / suitable_dinner` dari dataset v2.
- Guardrail `is_recommendable_food`, `ingredient_only_flag`, `raw_ingredient_flag`, dan kolom halal v2.
- **Variety Penalty** — makanan yang sudah dipakai di hari sebelumnya mendapat penalti skor.
- Menghasilkan rencana makan **7 hari** (Hari 1–7, masing-masing Pagi / Siang / Malam).

In [7]:
# ── Load model (skip jika sudah ada di memori) ───────────────────────────────
loaded_model = tf.keras.models.load_model(
    'nutrimatch_model.keras',
    custom_objects={
        'InteractionLayer': InteractionLayer,
        'asymmetric_allergy_loss': asymmetric_allergy_loss
    }
)
print("Model berhasil di-load.")

# ── Precompute food macro matrix (normalized) ────────────────────────────────
# food_max sudah didefinisikan di sel sebelumnya
# X_food_allergy juga sudah ada dari preprocessing
num_foods = len(food_subset)
_f_mac = food_subset[food_macro_cols].fillna(0).values.astype(np.float32) / food_max
_f_all = food_subset[allergy_cols_food].values.astype(np.float32)

# ── Kategori Role untuk dataset v2 ───────────────────────────────────────────
# v2 sudah punya food_category yang lebih rapi, jadi role bisa lebih presisi.
INVALID_FOOD_CATEGORIES = [
    'minyak', 'lemak', 'bumbu_sambal', 'bumbu', 'snack_dessert',
    'minuman', 'kue', 'jajanan'
]
INVALID_NAMES = [
    'tepung', 'mentah', 'kering', 'bubuk', 'sirup', 'kaldu', 'gula', 'garam',
    'kerupuk', 'keripik', 'beras', 'mentega', 'margarin', 'dideh', 'ampas',
    'abon', 'minyak', 'atom'
]
VALID_COMPLETE_NAMES = [
    'nasi goreng', 'mie goreng', 'soto', 'sup', 'bubur', 'burger', 'pizza',
    'nasi uduk', 'nasi kuning', 'nasi campur', 'nasi padang', 'nasi gemuk',
    'mie ayam', 'bakso', 'sate', 'spaghetti', 'macaroni',
    'lontong sayur', 'ketupat sayur', 'martabak'
]

def is_valid_for_role(row, role):
    fname_lower = str(row.get('food_name', '')).lower()
    cat_lower = str(row.get('food_category', '')).lower()
    cook_cat_lower = str(row.get('cooking_category', '')).lower()

    if any(invalid in cat_lower for invalid in INVALID_FOOD_CATEGORIES):
        return False
    if any(invalid in fname_lower for invalid in INVALID_NAMES):
        return False
    if cook_cat_lower == 'mentah_segar' and role in ['Complete', 'Karbo', 'Lauk']:
        return False

    is_complete_by_name = any(name in fname_lower for name in VALID_COMPLETE_NAMES)
    is_complete_by_cat = 'berkuah' in cat_lower
    if role == 'Complete':
        return is_complete_by_cat or is_complete_by_name
    if is_complete_by_cat or is_complete_by_name:
        return False

    words = fname_lower.replace('-', ' ').split()

    if role == 'Karbo':
        if 'karbohidrat_pokok' in cat_lower:
            return True
        return any(k in words for k in ['nasi', 'mie', 'roti', 'pasta', 'kentang', 'umbi', 'singkong', 'ubi', 'sereal', 'oat', 'gandum']) and cat_lower not in ['sayuran', 'buah', 'lauk_hewani', 'lauk_nabati']

    if role == 'Lauk':
        if cat_lower in ['lauk_hewani', 'lauk_nabati']:
            return True
        return any(l in words for l in ['daging', 'ayam', 'ikan', 'seafood', 'tahu', 'tempe', 'telur', 'kacang', 'sapi', 'kambing', 'udang', 'cumi']) and cat_lower not in ['sayuran', 'buah', 'karbohidrat_pokok']

    if role == 'Sayur':
        if cat_lower in ['sayuran', 'buah', 'tumis']:
            return True
        return any(s in words for s in ['sayur', 'bayam', 'kangkung', 'brokoli', 'wortel', 'tomat', 'kol', 'sawi', 'selada', 'jamur', 'salad', 'apel', 'pisang', 'jeruk', 'pepaya']) and cat_lower not in ['lauk_hewani', 'lauk_nabati', 'karbohidrat_pokok']

    return False

# ── Fungsi Rekomendasi Combo Meal ────────────────────────────────────────────
def get_combo_meal_recommendations(
    meal_type: str,          # 'breakfast' | 'lunch' | 'dinner'
    target_macros: list,     # [cal, prot, fat, carb] untuk total SATU sesi
    user_allergy_list: list,
    used_foods: set,         
    variety_penalty: float = 0.15,
) -> dict:
    """
    Menghasilkan 1 set Combo Meal (Karbo + Lauk + Sayur/Pelengkap) 
    berdasarkan pemecahan rasio target makro.
    """
    # 1. Alokasi rasio target per komponen (Bisa di-tuning)
    # Target: [Kalori, Protein, Lemak, Karbohidrat]
    target_k = [target_macros[0]*0.45, target_macros[1]*0.10, target_macros[2]*0.10, target_macros[3]*0.60] # Karbo
    target_l = [target_macros[0]*0.40, target_macros[1]*0.75, target_macros[2]*0.70, target_macros[3]*0.10] # Lauk
    target_s = [target_macros[0]*0.15, target_macros[1]*0.15, target_macros[2]*0.20, target_macros[3]*0.30] # Sayur

    u_all = np.array(user_allergy_list, dtype=np.float32)
    u_all_batch = np.tile(u_all, (num_foods, 1))
    
    has_allergy = np.any(np.logical_and(u_all_batch == 1, _f_all == 1), axis=1)

    # Helper untuk memprediksi skor massal
    def predict_scores(t_macro):
        u_mac = np.array(t_macro, dtype=np.float32) / user_max
        u_mac_batch = np.tile(u_mac, (num_foods, 1))
        scores, _ = loaded_model.predict([u_mac_batch, u_all_batch, _f_mac, _f_all], batch_size=256, verbose=0)
        return scores.flatten()

    scores_k = predict_scores(target_k)
    scores_l = predict_scores(target_l)
    scores_s = predict_scores(target_s)

    meal_col_map = {'breakfast': 'suitable_breakfast', 'lunch': 'suitable_lunch', 'dinner': 'suitable_dinner'}
    suitable_col = meal_col_map.get(meal_type, None)

    k_cands, l_cands, s_cands = [], [], []

    for i in range(num_foods):
        if has_allergy[i]: continue
        if suitable_col and food_subset.iloc[i].get(suitable_col, 1) == 0: continue
        
        row = food_subset.iloc[i]
        fname = row['food_name']
        cat = str(row.get('food_category', '')).lower()
        cal = row['calories_100g']
        # prot = row['protein_100g']
        # carb = row['carbohydrate_100g']
        
        if cal <= 0: continue

        # # HARD FILTER: Buang minyak/lemak/mentega murni & bahan dasar (bumbu)
        # if 'minyak' in fname.lower() or 'mentega' in fname.lower() or 'lemak' in fname.lower() or row.get('base_ingredient', 0) == 1:
        #     continue

        # # Klasifikasi Heuristik Fleksibel
        # is_karbo = ('pokok' in cat) or ('nasi' in fname.lower()) or ('mie' in fname.lower()) or (carb > 25 and prot < 10)
        # is_lauk  = ('lauk' in cat) or ('daging' in cat) or ('ikan' in cat) or ('ayam' in fname.lower()) or (prot > 10)
        # is_sayur = ('sayur' in cat) or ('buah' in cat) or (cal < 80 and not is_lauk and not is_karbo)

        penalty = variety_penalty if fname in used_foods else 0

        # Alokasikan kandidat berdasarkan role dataset v2
        if is_valid_for_role(row, 'Karbo'):
            k_cands.append((i, fname, scores_k[i] - penalty, row))
        elif is_valid_for_role(row, 'Lauk'):
            l_cands.append((i, fname, scores_l[i] - penalty, row))
        elif is_valid_for_role(row, 'Sayur'):
            s_cands.append((i, fname, scores_s[i] - penalty, row))

        # if is_karbo: k_cands.append((i, fname, scores_k[i] - penalty, row))
        # elif is_lauk: l_cands.append((i, fname, scores_l[i] - penalty, row))
        # elif is_sayur: s_cands.append((i, fname, scores_s[i] - penalty, row))
        # else: s_cands.append((i, fname, scores_s[i] - penalty, row)) # Fallback ke pelengkap

    # Ambil rank 1 dari tiap kategori
    k_cands.sort(key=lambda x: x[2], reverse=True)
    l_cands.sort(key=lambda x: x[2], reverse=True)
    s_cands.sort(key=lambda x: x[2], reverse=True)

    combo = {}
    for role, target_m, cands in zip(['Karbo', 'Lauk', 'Sayur'], [target_k, target_l, target_s], [k_cands, l_cands, s_cands]):
        if cands:
            best = cands[0]
            grams = (target_m[0] / best[3]['calories_100g']) * 100
            # Batasi porsi ekstrim
            grams = min(max(grams, 50), 300) 
            
            combo[role] = {
                'food_name': best[1],
                'grams': grams,
                'cal': (best[3]['calories_100g'] / 100) * grams,
                'prot': (best[3]['protein_100g'] / 100) * grams,
                'fat': (best[3]['fat_100g'] / 100) * grams,
                'carb': (best[3]['carbohydrate_100g'] / 100) * grams,
                'score': best[2]
            }
    return combo

# ── Fungsi rekomendasi per sesi makan ────────────────────────────────────────
def get_meal_recommendations(
    meal_name: str,
    meal_type: str,          # 'breakfast' | 'lunch' | 'dinner'
    target_macros: list,     # [cal, prot, fat, carb] untuk sesi ini
    user_allergy_list: list,
    used_foods: set,         # nama makanan yang sudah dipakai hari-hari sebelumnya
    top_k: int = 2,
    variety_penalty: float = 0.15,
) -> list:
    """
    Mengembalikan list dict rekomendasi makanan untuk satu sesi makan.
    Baru: filter meal-time & variety penalty.
    """
    u_mac = np.array(target_macros, dtype=np.float32) / user_max
    u_all = np.array(user_allergy_list, dtype=np.float32)

    u_mac_batch = np.tile(u_mac, (num_foods, 1))
    u_all_batch = np.tile(u_all, (num_foods, 1))

    match_scores, _ = loaded_model.predict(
        [u_mac_batch, u_all_batch, _f_mac, _f_all],
        batch_size=128, verbose=0
    )

    # Guardrail alergi
    has_allergy = np.any(
        np.logical_and(u_all_batch == 1, _f_all == 1), axis=1
    )

    # Kolom filter meal-time
    meal_col_map = {
        'breakfast': 'suitable_breakfast',
        'lunch':     'suitable_lunch',
        'dinner':    'suitable_dinner',
    }
    suitable_col = meal_col_map.get(meal_type, None)

    results = []
    target_cal = target_macros[0]

    for i in range(num_foods):
        # 1. Guardrail alergi
        if has_allergy[i]:
            continue
        # 2. Filter waktu makan
        if suitable_col and food_subset.iloc[i].get(suitable_col, 1) == 0:
            continue
        row = food_subset.iloc[i]
        if not any(is_valid_for_role(row, role) for role in ['Complete', 'Karbo', 'Lauk', 'Sayur']):
            continue
        # 3. Hitung porsi
        cal_100g = row['calories_100g']
        if cal_100g <= 0:
            continue

        ideal_grams = (target_cal / cal_100g) * 100
        ideal_prot  = (row['protein_100g']       / 100) * ideal_grams
        ideal_fat   = (row['fat_100g']           / 100) * ideal_grams
        ideal_carb  = (row['carbohydrate_100g']  / 100) * ideal_grams

        score = float(match_scores[i][0])

        # 4. Variety penalty jika makanan sudah dipakai sebelumnya
        fname = row['food_name']
        if fname in used_foods:
            score -= variety_penalty

        results.append({
            'food_name':   fname,
            'ideal_grams': ideal_grams,
            'ideal_prot':  ideal_prot,
            'ideal_fat':   ideal_fat,
            'ideal_carb':  ideal_carb,
            'match_score': score,
        })

    results = sorted(results, key=lambda x: x['match_score'], reverse=True)
    return results[:top_k]


# ── Fungsi 7-Day Meal Plan ────────────────────────────────────────────────────
MEAL_RATIOS = {
    'breakfast': ('Sarapan Pagi (25%)',  0.25),
    'lunch':     ('Makan Siang (40%)',   0.40),
    'dinner':    ('Makan Malam (35%)',   0.35),
}

DAY_NAMES = [
    'Senin', 'Selasa', 'Rabu', 'Kamis', 'Jumat', 'Sabtu', 'Minggu'
]

def generate_7day_meal_plan(
    daily_macros: list,
    user_allergy_list: list,
    top_k: int = 2,
    variety_penalty: float = 0.15,
    start_date: str = None,
):
    """
    Membuat rencana makan 7 hari.
    
    Parameters
    ----------
    daily_macros      : [total_cal, prot_g, fat_g, carb_g] per hari
    user_allergy_list : [gluten, dairy, nuts, peanut, seafood, egg, soy, celery]
    top_k             : jumlah opsi makanan per sesi
    variety_penalty   : pengurangan skor untuk makanan berulang
    start_date        : 'YYYY-MM-DD' (opsional); default = hari ini
    """
    if start_date:
        base_date = datetime.strptime(start_date, '%Y-%m-%d')
    else:
        base_date = datetime.today()

    allergy_labels = ['Gluten','Dairy','Nuts','Peanut','Seafood','Egg','Soy','Celery']
    active_allergies = [allergy_labels[i] for i, v in enumerate(user_allergy_list) if v == 1]

    header = (
        "\n" + "=" * 60 + "\n"
        f"  🗓️  RENCANA MAKAN 7 HARI — NutriMatch\n"
        + "=" * 60 + "\n"
        f"  Target Kalori Harian : {daily_macros[0]} kcal\n"
        f"  Protein / Lemak / Karbo : {daily_macros[1]}g / {daily_macros[2]}g / {daily_macros[3]}g\n"
        f"  Alergi Aktif : {', '.join(active_allergies) if active_allergies else 'Tidak ada'}\n"
        + "=" * 60
    )
    print(header)

    used_foods: set = set()

    for day_idx in range(7):
        current_date = base_date + timedelta(days=day_idx)
        day_label = DAY_NAMES[current_date.weekday()]
        date_str  = current_date.strftime('%d %b %Y')

        print(f"\n{'─'*60}")
        print(f"  📅 Hari {day_idx+1} — {day_label}, {date_str}")
        print(f"{'─'*60}")

        day_used: set = set()  # makanan yg dipakai hari ini (cegah duplikat antar sesi)

        for meal_key, (meal_label, ratio) in MEAL_RATIOS.items():
            meal_macros = [m * ratio for m in daily_macros]
            target_cal  = meal_macros[0]

            print(f"\n  🍽️  {meal_label} (Target: {target_cal:.0f} kcal)")

            recs = get_meal_recommendations(
                meal_label, meal_key, meal_macros,
                user_allergy_list,
                used_foods=used_foods | day_used,
                top_k=top_k,
                variety_penalty=variety_penalty,
            )

            if not recs:
                print("     ⚠️  Tidak ada makanan yang aman dari alergi di database.")
                continue

            for rank, res in enumerate(recs, 1):
                print(f"     Opsi {rank}: {res['food_name'].title()}")
                print(f"       Porsi      : {res['ideal_grams']:.0f} gram")
                print(f"       Gizi       : "
                      f"{res['ideal_prot']:.1f}g Protein | "
                      f"{res['ideal_fat']:.1f}g Lemak | "
                      f"{res['ideal_carb']:.1f}g Karbo")
                print(f"       Match Score: {res['match_score']*100:.2f}%")

            # Tandai opsi terbaik sebagai 'sudah dipakai' untuk variety
            day_used.add(recs[0]['food_name'])

        # Tambahkan semua makanan hari ini ke pool variety global
        used_foods |= day_used

    print("\n" + "=" * 60)
    print("  ✅ Rencana makan 7 hari selesai dibuat!")
    print("=" * 60)


def generate_7day_combo_plan(daily_macros, user_allergy_list, variety_penalty=0.15, start_date=None):
    base_date = datetime.strptime(start_date, '%Y-%m-%d') if start_date else datetime.today()
    allergy_labels = ['Gluten','Dairy','Nuts','Peanut','Seafood','Egg','Soy','Celery']
    active_allergies = [allergy_labels[i] for i, v in enumerate(user_allergy_list) if v == 1]

    print("\n" + "=" * 60)
    print("  🗓️  RENCANA MAKAN 7 HARI (COMBO MEAL) — NutriMatch")
    print("=" * 60)
    print(f"  Target Kalori Harian : {daily_macros[0]} kcal")
    print(f"  Protein / Lemak / Karbo : {daily_macros[1]}g / {daily_macros[2]}g / {daily_macros[3]}g")
    print(f"  Alergi Aktif : {', '.join(active_allergies) if active_allergies else 'Tidak ada'}")
    print("=" * 60)

    used_foods = set()

    for day_idx in range(7):
        current_date = base_date + timedelta(days=day_idx)
        print(f"\n{'─'*60}\n  📅 Hari {day_idx+1} — {DAY_NAMES[current_date.weekday()]}, {current_date.strftime('%d %b %Y')}\n{'─'*60}")

        day_used = set()

        for meal_key, (meal_label, ratio) in MEAL_RATIOS.items():
            meal_macros = [m * ratio for m in daily_macros]
            print(f"\n  🍽️  {meal_label} (Target: {meal_macros[0]:.0f} kcal)")

            combo = get_combo_meal_recommendations(meal_key, meal_macros, user_allergy_list, used_foods | day_used, variety_penalty)

            if not combo:
                print("     ⚠️ Tidak ada makanan yang sesuai/aman.")
                continue

            total_cal = 0
            avg_score = 0
            for role in ['Karbo', 'Lauk', 'Sayur']:
                if role in combo:
                    item = combo[role]
                    total_cal += item['cal']
                    avg_score += item['score']
                    day_used.add(item['food_name'])
                    print(f"     [{role}] {item['food_name'].title()}")
                    print(f"             {item['grams']:>5.0f}g | {item['cal']:>5.0f} kcal | P:{item['prot']:>4.1f}g | L:{item['fat']:>4.1f}g | K:{item['carb']:>4.1f}g")
            
            print(f"     --------------------------------------------------")
            print(f"     🔥 Estimasi Sesi: {total_cal:.0f} kcal (Avg Match: {(avg_score/len(combo))*100:.1f}%)")

        used_foods |= day_used

    print("\n" + "=" * 60 + "\n  ✅ Rencana makan 7 hari (Combo) selesai dibuat!\n" + "=" * 60)

# ── TEST CASE 1: User biasa tanpa alergi ─────────────────────────────────────
print("\n" + "#" * 60)
print("# TEST CASE 1: User Biasa (Tanpa Alergi)")
print("#" * 60)
generate_7day_meal_plan(
    daily_macros      = [1800, 100, 50, 200],
    user_allergy_list = [0, 0, 0, 0, 0, 0, 0, 0],
    top_k             = 2,
)

# ── TEST CASE 2: User alergi seafood & telur ──────────────────────────────────
print("\n" + "#" * 60)
print("# TEST CASE 2: User Alergi Seafood & Telur")
print("#" * 60)
generate_7day_meal_plan(
    daily_macros      = [2200, 120, 60, 250],
    user_allergy_list = [0, 0, 0, 0, 1, 1, 0, 0],
    top_k             = 2,
    start_date        = '2025-06-02',
)

# ── TEST CASES ───────────────────────────────────────────────────────────────
print("\n" + "#" * 60 + "\n# TEST CASE 1: User Biasa (Tanpa Alergi)\n" + "#" * 60)
generate_7day_combo_plan([1800, 100, 50, 200], [0, 0, 0, 0, 0, 0, 0, 0])

print("\n" + "#" * 60 + "\n# TEST CASE 2: User Alergi Seafood & Telur\n" + "#" * 60)
generate_7day_combo_plan([2200, 120, 60, 250], [0, 0, 0, 0, 1, 1, 0, 0], start_date='2025-06-02')

Model berhasil di-load.

############################################################
# TEST CASE 1: User Biasa (Tanpa Alergi)
############################################################

  🗓️  RENCANA MAKAN 7 HARI — NutriMatch
  Target Kalori Harian : 1800 kcal
  Protein / Lemak / Karbo : 100g / 50g / 200g
  Alergi Aktif : Tidak ada

────────────────────────────────────────────────────────────
  📅 Hari 1 — Kamis, 04 Jun 2026
────────────────────────────────────────────────────────────

  🍽️  Sarapan Pagi (25%) (Target: 450 kcal)
     Opsi 1: Bagea Kelapa Asin
       Porsi      : 100 gram
       Gizi       : 3.2g Protein | 13.9g Lemak | 78.1g Karbo
       Match Score: 89.32%
     Opsi 2: Telur Terubuk
       Porsi      : 106 gram
       Gizi       : 32.8g Protein | 29.6g Lemak | 10.6g Karbo
       Match Score: 87.50%

  🍽️  Makan Siang (40%) (Target: 720 kcal)
     Opsi 1: Ikan Asin Pepetek Goreng
       Porsi      : 110 gram
       Gizi       : 44.6g Protein | 60.0g Lemak | 0.0g Karb

## 7. (Opsional) Export Rencana Makan ke CSV

In [8]:
def export_meal_plan_csv(
    daily_macros: list,
    user_allergy_list: list,
    top_k: int = 2,
    variety_penalty: float = 0.15,
    start_date: str = None,
    output_path: str = 'meal_plan_7days.csv',
):
    """Menyimpan rencana makan 7 hari ke file CSV."""
    if start_date:
        base_date = datetime.strptime(start_date, '%Y-%m-%d')
    else:
        base_date = datetime.today()

    rows = []
    used_foods: set = set()

    for day_idx in range(7):
        current_date = base_date + timedelta(days=day_idx)
        day_label    = DAY_NAMES[current_date.weekday()]
        date_str     = current_date.strftime('%Y-%m-%d')
        day_used: set = set()

        for meal_key, (meal_label, ratio) in MEAL_RATIOS.items():
            meal_macros = [m * ratio for m in daily_macros]
            recs = get_meal_recommendations(
                meal_label, meal_key, meal_macros,
                user_allergy_list,
                used_foods=used_foods | day_used,
                top_k=top_k,
                variety_penalty=variety_penalty,
            )
            for rank, res in enumerate(recs, 1):
                rows.append({
                    'hari_ke'      : day_idx + 1,
                    'hari'         : day_label,
                    'tanggal'      : date_str,
                    'sesi'         : meal_label,
                    'opsi'         : rank,
                    'makanan'      : res['food_name'].title(),
                    'porsi_gram'   : round(res['ideal_grams'], 1),
                    'protein_g'    : round(res['ideal_prot'], 1),
                    'lemak_g'      : round(res['ideal_fat'], 1),
                    'karbo_g'      : round(res['ideal_carb'], 1),
                    'match_score_%': round(res['match_score'] * 100, 2),
                })
            if recs:
                day_used.add(recs[0]['food_name'])
        used_foods |= day_used

    df_out = pd.DataFrame(rows)
    df_out.to_csv(output_path, index=False)
    print(f"✅ Rencana makan tersimpan di '{output_path}' ({len(df_out)} baris)")
    return df_out


# Contoh export
df_plan = export_meal_plan_csv(
    daily_macros      = [1800, 100, 50, 200],
    user_allergy_list = [0, 0, 0, 0, 0, 0, 0, 0],
    output_path       = 'meal_plan_7days.csv',
)
df_plan.head(10)

✅ Rencana makan tersimpan di 'meal_plan_7days.csv' (42 baris)


,hari_ke,hari,tanggal,sesi,opsi,makanan,porsi_gram,protein_g,lemak_g,karbo_g,match_score_%
0,1,Kamis,2026-06-04,Sarapan Pagi (25%),1,Bagea Kelapa Asin,100.0,3.2,13.9,78.1,89.32
1,1,Kamis,2026-06-04,Sarapan Pagi (25%),2,Telur Terubuk,105.9,32.8,29.6,10.6,87.50
2,1,Kamis,2026-06-04,Makan Siang (40%),1,Ikan Asin Pepetek Goreng,110.4,44.6,60.0,0.0,95.47
3,1,Kamis,2026-06-04,Makan Siang (40%),2,Biji Jambu Mete Goreng,114.5,24.6,64.8,22.7,95.14
4,1,Kamis,2026-06-04,Makan Malam (35%),1,Biji Jambu Mete Goreng,100.2,21.5,56.7,19.8,95.17
5,1,Kamis,2026-06-04,Makan Malam (35%),2,Kacang Metebiji Jambu Monyet Goreng,100.2,20.4,56.4,19.8,95.10
6,2,Jumat,2026-06-05,Sarapan Pagi (25%),1,Telur Terubuk,105.9,32.8,29.6,10.6,87.50
7,2,Jumat,2026-06-05,Sarapan Pagi (25%),2,Ledre Pisang,111.1,5.3,5.7,94.3,87.32
8,2,Jumat,2026-06-05,Makan Siang (40%),1,Kacang Metebiji Jambu Monyet Goreng,114.5,23.4,64.4,22.7,95.07
9,2,Jumat,2026-06-05,Makan Siang (40%),2,Ikan Mujair Dendeng Goreng,120.4,89.5,32.4,11.1,94.69
